## Extraction of Data from MODIS Vegetation Indices Dataset (GEE) ##

In [2]:
# Import Google Earth Engine API and Initialize it. 
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project="ey-data-and-ai-challenge")

In [3]:
# Read coordinates and date from water quality training dataset, drop given features.

wq_df = pd.read_csv('../data/water_quality_training_dataset.csv')
wq_df = wq_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
wq_df['id'] = wq_df.index
wq_df['Sample Date'] = pd.to_datetime(wq_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
wq_df.head()

,Latitude,Longitude,Sample Date,id
0,-28.760833,17.730278,2011-01-02,0
1,-26.861111,28.884722,2011-01-03,1
2,-26.450000,28.085833,2011-01-03,2
3,-27.671111,27.236944,2011-01-03,3
4,-27.356667,27.286389,2011-01-03,4


In [14]:
# Convert Coordinates and given date to ee.Features for use in batch export.

features = []

for index, row in wq_df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(250), #add a 100m buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=4)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=4)).strftime('%Y-%m-%d')
        }
    )
    features.append(feat)

fc = ee.FeatureCollection(features)         # create feature collection with features

In [15]:
modisvi_collection = ee.ImageCollection("MODIS/061/MOD13Q1").select(['NDVI', 'EVI'])       # Selecting vegetation indices

In [16]:
def extract_median_values(feat):
    collection = modisvi_collection.filterDate(feat.get('start_date'), feat.get('end_date'))
    img = collection.reduce(ee.Reducer.median()) # reduce image collection into a single image

    modisvi_col = img.reduceRegions(collection=ee.FeatureCollection([feat]), reducer=ee.Reducer.first(), scale = 250)
    
    return modisvi_col.first()

In [17]:
fc_mapped = fc.map(extract_median_values)

In [18]:
# Process data and export to Google Drive

task = ee.batch.Export.table.toDrive(
    collection=fc_mapped,
    description="modisVI_csv_export",
    fileNamePrefix= "modisVI_features_training",
    fileFormat='CSV'
)
task.start()

In [19]:
modis_df = pd.read_csv("../data/modisVI_features_training.csv")

# Drop irrelevant columns
modis_df.drop(columns=[".geo", "system:index", "end_date", "start_date"], inplace=True)

modis_df = modis_df.merge(wq_df, on='id', how='left')
modis_df.drop(columns=['id'], inplace=True)
modis_df

,EVI_median,NDVI_median,Latitude,Longitude,Sample Date
0,241.0,636.0,-28.760833,17.730278,2011-01-02
1,4643.0,7656.0,-26.861111,28.884722,2011-01-03
2,3984.0,6276.0,-26.450000,28.085833,2011-01-03
3,2217.0,3957.0,-27.671111,27.236944,2011-01-03
4,4136.0,7396.0,-27.356667,27.286389,2011-01-03
...,...,...,...,...,...
9314,2783.0,5093.0,-27.527500,30.858056,2015-12-23
9315,3951.0,5845.5,-26.861111,28.884722,2015-12-23
9316,1744.0,2921.5,-26.984722,26.632278,2015-12-23
9317,2115.0,3917.5,-27.935000,26.126667,2015-12-23


In [20]:
modis_df.to_csv("../data/modisVI_features_training.csv")

# Repeat for Validation Set

In [21]:
val_df = pd.read_csv("../data/submission_template.csv")
val_df['id'] = val_df.index
val_df['Sample Date'] = pd.to_datetime(val_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
val_df

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,id
0,-32.043333,27.822778,2014-09-01,NaN,NaN,NaN,0
1,-33.329167,26.077500,2015-09-16,NaN,NaN,NaN,1
2,-32.991639,27.640028,2015-05-07,NaN,NaN,NaN,2
3,-34.096389,24.439167,2012-02-07,NaN,NaN,NaN,3
4,-32.000556,28.581667,2014-10-01,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...
195,-33.771111,25.386667,2012-12-06,NaN,NaN,NaN,195
196,-33.185361,27.390750,2014-09-04,NaN,NaN,NaN,196
197,-32.043333,27.822778,2015-09-28,NaN,NaN,NaN,197
198,-33.001667,25.161389,2015-01-08,NaN,NaN,NaN,198


In [25]:

features_val = []

for index, row in val_df.iterrows():
    feat_val = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(250), #add a 10km buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=4)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=4)).strftime('%Y-%m-%d')
        }
    )
    features_val.append(feat_val)

fc_val = ee.FeatureCollection(features_val)

In [26]:
fc_mapped_val = fc_val.map(extract_median_values)

In [27]:
task = ee.batch.Export.table.toDrive(
    collection=fc_mapped_val,
    description="modisVI_val_csv_export",
    fileNamePrefix= "modisVI_features_validation",
    fileFormat='CSV'
)
task.start()

In [29]:
modis_val_df = pd.read_csv("../data/modisVI_features_validation.csv")

# Drop irrelevant columns
modis_val_df.drop(columns=[".geo", "system:index", "end_date", "start_date"], inplace=True)

modis_val_df = modis_val_df.merge(val_df, on='id', how='left')
modis_val_df.drop(columns=['id'], inplace=True)
modis_val_df

,EVI_median,NDVI_median,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,1628.0,3016.0,-32.043333,27.822778,2014-09-01,NaN,NaN,NaN
1,2933.0,7263.0,-33.329167,26.077500,2015-09-16,NaN,NaN,NaN
2,4032.0,7036.0,-32.991639,27.640028,2015-05-07,NaN,NaN,NaN
3,4701.0,7474.5,-34.096389,24.439167,2012-02-07,NaN,NaN,NaN
4,1498.0,4257.0,-32.000556,28.581667,2014-10-01,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
195,2710.0,4137.5,-33.771111,25.386667,2012-12-06,NaN,NaN,NaN
196,2763.5,5440.0,-33.185361,27.390750,2014-09-04,NaN,NaN,NaN
197,2148.0,4215.0,-32.043333,27.822778,2015-09-28,NaN,NaN,NaN
198,1029.0,1940.0,-33.001667,25.161389,2015-01-08,NaN,NaN,NaN


In [30]:
modis_val_df.to_csv("../data/modisVI_features_validation.csv")